# 2) Statistical models — Exercises

**Goals:** find the heating and cooling thresholds from data, quantify the weather leak, test how holidays should enter a regression, build the error correction for an intraday forecast, and put a number on the value of a better temperature forecast.

The setup cell loads the lecture's data with the honest temperature forecast, the `daytype`, `hour` and `month` columns, the lecture's model formula `LECTURE` and a helper `mae`. Demand in MW, temperatures in °C.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
train, test = df.loc[:"2024"], df.loc["2025"]
SPLINE = "bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
LECTURE = f"demand ~ C(hour):C(daytype) + C(month) + {SPLINE}"     # the lecture's temperature model


def mae(forecast, actual):
    return float(np.abs(np.asarray(forecast) - np.asarray(actual)).mean())

### Warm-ups

**1. Finding the thresholds**

The lecture's spline learned the temperature response without being told where heating and cooling start. Do it explicitly: for heating thresholds in [12, 14, 16, 18] °C and cooling thresholds in [20, 22, 24, 26] °C, fit `demand ~ C(hour):C(daytype) + C(month) + hdd + cdd` (hdd = max(threshold_h − T, 0), cdd = max(T − threshold_c, 0)) on **2023**, and score it on **2024** with the actual temperature. Write `best_thresholds()` returning the best pair and the table of validation MAEs.

In [2]:
def best_thresholds(heating=(12, 14, 16, 18), cooling=(20, 22, 24, 26)) -> tuple[tuple[int, int], pd.DataFrame]:
    """Heating and cooling thresholds (deg C) with the lowest 2024 MAE, and the full MAE table."""
    ### BEGIN SOLUTION
    fit, valid = df.loc["2023"], df.loc["2024"]
    table = pd.DataFrame(index=list(heating), columns=list(cooling), dtype=float)
    for th in heating:
        for tc in cooling:
            add = lambda d: d.assign(hdd=np.clip(th - d.temperature, 0, None), cdd=np.clip(d.temperature - tc, 0, None))
            model = smf.ols("demand ~ C(hour):C(daytype) + C(month) + hdd + cdd", add(fit)).fit()
            table.loc[th, tc] = mae(model.predict(add(valid)), valid.demand)
    best = table.stack().idxmin()
    return (int(best[0]), int(best[1])), table.rename_axis(index="heating", columns="cooling")
    ### END SOLUTION

In [3]:
best, table = best_thresholds()
assert best == (16, 22), "the data were generated with heating below 16 and cooling above 22 deg C"
print("best thresholds:", best)
table.round(1)

best thresholds: (16, 22)


cooling    20    22    24    26
heating                        
12       66.9  62.9  65.8  72.7
14       63.3  59.4  62.1  69.0
16       60.7  57.2  60.3  67.3
18       60.4  57.9  61.9  69.4

**2. How much does the weather leak flatter?**

Write `leak_optimism(formula)`: fit the formula on 2023–2024, score 2025 once with the temperature forecast and once with the actual temperature, and return both MAEs and how much better the leaky score looks (in %). Apply it to the lecture's model and to the calendar-only model.

In [4]:
def leak_optimism(formula: str) -> dict:
    """Honest and leaky 2025 MAE of a model, and the leak's apparent improvement in %."""
    ### BEGIN SOLUTION
    model = smf.ols(formula, train).fit()
    honest = mae(model.predict(test.assign(temperature=test.temperature_forecast)), test.demand)
    leaky = mae(model.predict(test), test.demand)
    return {"honest": honest, "leaky": leaky, "looks_better_by_%": 100 * (1 - leaky / honest)}
    ### END SOLUTION

In [5]:
weather = leak_optimism(LECTURE)
calendar = leak_optimism("demand ~ C(hour):C(daytype) + C(month)")
assert np.isclose(weather["honest"], 73.6, atol=0.1) and np.isclose(weather["leaky"], 58.6, atol=0.1)
assert np.isclose(calendar["looks_better_by_%"], 0.0), "a model without temperature cannot leak through it"
pd.DataFrame({"temperature model": weather, "calendar only": calendar}).round(1)

                   temperature model  calendar only
honest                          73.6          115.7
leaky                           58.6          115.7
looks_better_by_%               20.3            0.0

### Core

**3. How should holidays enter a regression?**

The lecture treats holidays as Sundays. Compare three versions of the lecture's model, fitted on 2023–2024 and scored on 2025 (honest temperature) — overall and on holidays only: (a) holidays ignored (they count as normal weekdays), (b) holidays as Sundays (the lecture), (c) holidays as a fourth day type of their own. Write `holiday_variants()`.

In [6]:
def holiday_variants() -> pd.DataFrame:
    """2025 MAE overall and on holidays for three ways of encoding holidays."""
    ### BEGIN SOLUTION
    dow_ = df.index.dayofweek
    plain = np.where(dow_ == 6, "sunday", np.where(dow_ == 5, "saturday", "workday"))
    variants = {
        "ignored": plain,
        "as Sundays (lecture)": df["daytype"].to_numpy(),
        "own day type": np.where(df.holiday, "holiday", plain),
    }
    rows = {}
    hol = test.holiday.to_numpy()
    for name, daytype in variants.items():
        d = df.assign(daytype=daytype)
        model = smf.ols(LECTURE, d.loc[:"2024"]).fit()
        t = d.loc["2025"]
        err = np.abs(model.predict(t.assign(temperature=t.temperature_forecast)).to_numpy() - t.demand.to_numpy())
        rows[name] = {"MAE all": err.mean(), "MAE holidays": err[hol].mean()}
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [7]:
variants = holiday_variants()
assert variants.loc["ignored", "MAE holidays"] > 2 * variants.loc["as Sundays (lecture)", "MAE holidays"]
assert np.isclose(variants.loc["as Sundays (lecture)", "MAE all"], 73.6, atol=0.1)
variants.round(1)

                      MAE all  MAE holidays
ignored                  78.4         200.0
as Sundays (lecture)     73.6          66.0
own day type             73.6          67.7

**4. An intraday forecast**

Write `corrected_forecast(base, known_error, phi, k)` — the lecture's correction, base + φᵏ · (error known k hours earlier). Then build an **intraday** forecast: issued at 06:00, for the hours 07:00–12:00 of the same day (k = 1 … 6). Compare its MAE on 2025 with the day-ahead forecast for the same hours.

In [8]:
def corrected_forecast(base: pd.Series, known_error: pd.Series, phi: float, k) -> pd.Series:
    """Model forecast plus the error known k hours earlier, shrunk by phi**k (k may vary by hour)."""
    ### BEGIN SOLUTION
    k = pd.Series(k, index=base.index) if np.ndim(k) else pd.Series(int(k), index=base.index)
    shifted = pd.Series([known_error.get(t - pd.Timedelta(hours=int(kk)), np.nan) for t, kk in zip(base.index, k)],
                        index=base.index)
    return base + phi ** k * shifted
    ### END SOLUTION

In [9]:
model = smf.ols(LECTURE, train).fit()
phi = (df.demand - model.predict(df)).loc[:"2024"].autocorr(1)
known_error = df.demand - model.predict(df)
morning = test.index[(test.index.hour >= 7) & (test.index.hour <= 12)]
base = model.predict(df.assign(temperature=df.temperature_forecast)).loc[morning]
intraday = corrected_forecast(base, known_error, phi, morning.hour - 6)       # issued at 06:00: k = 1 ... 6
toy = corrected_forecast(pd.Series([100.0], index=[pd.Timestamp("2024-01-01 05:00")]),
                         pd.Series([10.0], index=[pd.Timestamp("2024-01-01 03:00")]), 0.5, 2)
assert np.isclose(toy.iloc[0], 100 + 0.25 * 10)
result = pd.Series({"day-ahead": mae(base, df.demand.loc[morning]), "intraday from 06:00": mae(intraday, df.demand.loc[morning])})
assert result["intraday from 06:00"] < 0.9 * result["day-ahead"]
result.round(1)

day-ahead              75.9
intraday from 06:00    64.4
dtype: float64

**5. Interpretation**

Which holiday encoding wins, and why does "ignored" fail so badly on holidays? How large is the intraday gain for the morning hours, and why could a TSO not simply use the intraday forecast for the day-ahead market?

<!-- BEGIN SOLUTION -->
Treating holidays as **Sundays** wins: 66.0 MW on holidays against **200.0 MW** when they are ignored. A model that believes a holiday is a normal workday predicts a full working-day load profile on a day when offices and factories are closed — the error on those days is three times the usual one, and it lifts the overall MAE from 73.6 to 78.4 MW although holidays are only a few percent of the hours. Giving holidays a **day type of their own** is slightly worse (67.7 MW): with only about 25 holidays in the training years, the model must estimate a separate 24-hour profile from very few days, and that estimate is noisier than simply borrowing the Sunday profile, which is learned from about a hundred days. More flexibility is not better when there is too little data to support it.

The intraday forecast issued at 06:00 is 15 % better for the morning hours (64.4 against 75.9 MW), because it knows how far demand was off the model at 05:00 and that error is still alive one to six hours later. A TSO cannot use it for the day-ahead market because the day-ahead market closes at noon on the day before: at that time 06:00 on the delivery day lies 18 hours in the future. Intraday forecasts serve a different decision — trading in the intraday market and preparing balancing — which is exactly why both kinds of forecast exist side by side.
<!-- END SOLUTION -->

### Challenge

**6. What is a better weather forecast worth?**

Scale the temperature-forecast error: use temperature + s · (forecast − temperature) for s in [0, 0.25, 0.5, 1, 1.5, 2] (s = 0 is a perfect forecast, s = 1 the lecture's). Keep the model fitted on actual temperatures, score 2025 for each s, and report the demand MAE against the temperature MAE. Large errors can push a forecast outside the range the spline was fitted on (−12 to 40 °C), where it cannot extrapolate: clip the forecast temperatures to that range first — the model then treats an extreme forecast like the most extreme value it has learned. Roughly how many MW of demand error does each °C of temperature error cost?

In [10]:
def weather_value(scales=(0, 0.25, 0.5, 1, 1.5, 2)) -> pd.DataFrame:
    """2025 demand MAE (MW) for temperature forecasts with scaled errors."""
    ### BEGIN SOLUTION
    model = smf.ols(LECTURE, train).fit()
    error = test.temperature_forecast - test.temperature
    rows = {}
    for s in scales:
        temp = (test.temperature + s * error).clip(-12, 40)         # stay inside the spline's range
        rows[s] = {"temperature_MAE": float(np.abs(s * error).mean()),
                   "demand_MAE": mae(model.predict(test.assign(temperature=temp)), test.demand)}
    return pd.DataFrame(rows).T.rename_axis("error_scale")
    ### END SOLUTION

In [11]:
value = weather_value()
assert value.demand_MAE.is_monotonic_increasing
assert np.isclose(value.loc[1, "demand_MAE"], 73.6, atol=0.1) and np.isclose(value.loc[0, "demand_MAE"], 58.6, atol=0.1)
slope = np.polyfit(value.temperature_MAE.iloc[2:], value.demand_MAE.iloc[2:], 1)[0]
print(f"beyond small errors, each extra 1 deg C of temperature MAE costs about {slope:.0f} MW of demand MAE")
value.round(2)

beyond small errors, each extra 1 deg C of temperature MAE costs about 22 MW of demand MAE


             temperature_MAE  demand_MAE
error_scale                             
0.00                    0.00       58.64
0.25                    0.32       59.63
0.50                    0.63       62.89
1.00                    1.27       73.56
1.50                    1.90       87.97
2.00                    2.53      104.55

**7. Interpretation**

Describe the curve: is the cost of temperature error linear, and why not at small errors? What does the number suggest a TSO should be willing to pay for better weather forecasts — and what else would you need to know to answer that properly?

<!-- BEGIN SOLUTION -->
The curve is **convex**. Small temperature errors cost almost nothing: an error of 0.32 °C adds only 1 MW to the 58.6 MW floor that even a perfect weather forecast leaves. Large errors cost more and more: from 1.27 to 2.53 °C the demand MAE climbs from 73.6 to 104.6 MW. The reason is that independent errors add in squares, not linearly — the demand error is roughly √(noise² + (sensitivity × temperature error)²). While the temperature error is small compared with demand's own noise, it hardly shows; once it dominates, every further degree costs the full sensitivity. The fitted slope of about 22 MW per °C is an average over a curve that keeps steepening.

For a TSO the relevant number is the marginal one at its current forecast quality: here, halving the temperature error (1.27 → 0.63 °C) would cut the demand error from 73.6 to 62.9 MW, about 15 %. Whether that is worth paying for depends on what a MW of forecast error costs — in balancing energy, in reserves held, in imbalance charges — and on how often the errors fall in tight hours where they are expensive. That conversion from forecast error to money is the subject of lesson 5.
<!-- END SOLUTION -->